In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "../data"

genuine = pd.read_csv(os.path.join(DATA_PATH, "genuine_accounts/users.csv"))
social_spambots_1 = pd.read_csv(os.path.join(DATA_PATH, "social_spambots_1/users.csv"))
social_spambots_2 = pd.read_csv(os.path.join(DATA_PATH, "social_spambots_2/users.csv"))
social_spambots_3 = pd.read_csv(os.path.join(DATA_PATH, "social_spambots_3/users.csv"))
traditional_spambots_1 = pd.read_csv(os.path.join(DATA_PATH, "traditional_spambots_1/users.csv"))
fake_followers = pd.read_csv(os.path.join(DATA_PATH, "fake_followers/users.csv"))

genuine["label"] = 0
social_spambots_1["label"] = 1
social_spambots_2["label"] = 1
social_spambots_3["label"] = 1
traditional_spambots_1["label"] = 1
fake_followers["label"] = 1

df = pd.concat([
    genuine, social_spambots_1, social_spambots_2,
    social_spambots_3, traditional_spambots_1, fake_followers
], ignore_index=True)

print("Loaded:", df.shape)

Loaded: (12737, 43)


In [2]:

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

REFERENCE_DATE = pd.Timestamp("2017-01-01", tz="UTC")

df["account_age_days"] = (REFERENCE_DATE - df["created_at"]).dt.days

df["account_age_days"] = df["account_age_days"].clip(lower=0)

print("Account age sample:")
print(df["account_age_days"].describe().round(1))

C:\Users\SEKINA\AppData\Local\Temp\ipykernel_19224\1753461492.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


Account age sample:
count    11737.0
mean      1558.3
std        554.6
min        621.0
25%       1043.0
50%       1433.0
75%       1810.0
max       3631.0
Name: account_age_days, dtype: float64


In [3]:
# --- Ratio features ---

# follower/friend ratio: high = popular real person, near-zero = follow-spam bot
# We add 1 to the bottom number to avoid dividing by zero
# (an account following 0 people would cause a crash without this)
df["follower_friend_ratio"] = df["followers_count"] / (df["friends_count"] + 1)

# tweet frequency: how many tweets per day since account creation
df["tweet_frequency"] = df["statuses_count"] / (df["account_age_days"] + 1)

# favourites per tweet: how engaged is this account with other content?
# bots almost never like anything — this was our 200x signal from earlier
df["favourites_per_tweet"] = df["favourites_count"] / (df["statuses_count"] + 1)

# listed per follower: being added to lists is a sign of a trusted, real account
df["listed_per_follower"] = df["listed_count"] / (df["followers_count"] + 1)

print("New ratio columns added.")
print(df[["follower_friend_ratio", "tweet_frequency", 
          "favourites_per_tweet", "listed_per_follower"]].describe().round(3))

New ratio columns added.
       follower_friend_ratio  tweet_frequency  favourites_per_tweet  \
count              12737.000        11737.000             12737.000   
mean                   1.596            3.062                 0.192   
std                   39.781           10.236                 1.227   
min                    0.000            0.000                 0.000   
25%                    0.052            0.030                 0.000   
50%                    0.157            0.060                 0.000   
75%                    0.721            1.524                 0.035   
max                 3595.750          377.267                95.287   

       listed_per_follower  
count            12737.000  
mean                 0.012  
std                  0.049  
min                  0.000  
25%                  0.000  
50%                  0.000  
75%                  0.004  
max                  1.000  


In [4]:

df["has_profile_image"] = (~df["default_profile_image"].fillna(True).astype(bool)).astype(int)

df["has_description"] = (df["description"].fillna("").str.strip().str.len() > 0).astype(int)


df["has_url"] = df["url"].notna().astype(int)

df["has_location"] = df["location"].fillna("").str.strip().str.len() > 0
df["has_location"] = df["has_location"].astype(int)

print("Binary flag columns added.")
print(df[["has_profile_image", "has_description", 
          "has_url", "has_location"]].value_counts())

Binary flag columns added.
has_profile_image  has_description  has_url  has_location
0                  0                0        0               4606
                   1                0        1               4102
                                    1        1               1259
                                    0        0               1227
                   0                0        1                754
                   1                1        0                423
                   0                1        0                286
                                             1                 80
Name: count, dtype: int64


In [5]:

FEATURE_COLUMNS = [
    # Direct columns
    "statuses_count",
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    # Engineered ratios
    "follower_friend_ratio",
    "tweet_frequency",
    "favourites_per_tweet",
    "listed_per_follower",
    # Binary flags
    "has_profile_image",
    "has_description",
    "has_url",
    "has_location",
    "account_age_days",
]

features_df = df[FEATURE_COLUMNS + ["label"]].copy()

print("Final feature table shape:", features_df.shape)
print("\nMissing values in feature table:")
print(features_df.isnull().sum())

Final feature table shape: (12737, 15)

Missing values in feature table:
statuses_count              0
followers_count             0
friends_count               0
favourites_count            0
listed_count                0
follower_friend_ratio       0
tweet_frequency          1000
favourites_per_tweet        0
listed_per_follower         0
has_profile_image           0
has_description             0
has_url                     0
has_location                0
account_age_days         1000
label                       0
dtype: int64


In [6]:

features_df.groupby("label")[FEATURE_COLUMNS].mean().round(3)

,statuses_count,followers_count,friends_count,favourites_count,listed_count,follower_friend_ratio,tweet_frequency,favourites_per_tweet,listed_per_follower,has_profile_image,has_description,has_url,has_location,account_age_days
label,,,,,,,,,,,,,,
0,16958.22,1393.220,633.242,4669.620,19.497,3.352,9.181,0.678,0.015,0.0,0.891,0.364,0.680,1832.119
1,845.50,399.582,589.835,23.383,2.048,0.937,0.490,0.010,0.011,0.0,0.423,0.084,0.414,1443.140


In [7]:

features_df = features_df.drop(columns=["has_profile_image"])
FEATURE_COLUMNS = [col for col in FEATURE_COLUMNS if col != "has_profile_image"]

print("Features remaining:", FEATURE_COLUMNS)

Features remaining: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'follower_friend_ratio', 'tweet_frequency', 'favourites_per_tweet', 'listed_per_follower', 'has_description', 'has_url', 'has_location', 'account_age_days']


In [8]:

features_df.to_csv("../data/features.csv", index=False)

print("Saved to data/features.csv")

Saved to data/features.csv
